In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

In [2]:
merged = pd.read_csv(
    processed_path / "merged_raw.csv"
)

merged["timestamp"] = pd.to_datetime(
    merged["timestamp"],
    errors="coerce"
)

print(merged.shape)
print(merged.columns.tolist())

(65535, 26)
['Patient', 'timestamp', 'glucose_value', 'meal_type', 'meal_carbs', 'bolus_type', 'bolus_dose', 'basal_value', 'temp_basal_value', 'exercise_type', 'exercise_duration', 'exercise_intensity', 'gsr', 'skin_temperature', 'acceleration', 'is_sleeping', 'sleep_quality', 'is_working', 'work_intensity', 'stressor_type', 'stressor_description', 'hypo_event', 'illness_type', 'illness_description', 'illness_event', 'fingerstick_glucose']


In [3]:
merged = merged.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

In [4]:
print(merged[[
    "Patient",
    "timestamp",
    "glucose_value"
]].head(10))

           Patient           timestamp  glucose_value
0  540-ws-training 2027-05-19 11:36:29             76
1  540-ws-training 2027-05-19 11:41:29             72
2  540-ws-training 2027-05-19 11:46:29             68
3  540-ws-training 2027-05-19 11:51:29             65
4  540-ws-training 2027-05-19 11:56:29             63
5  540-ws-training 2027-05-19 12:01:29             66
6  540-ws-training 2027-05-19 12:06:29             71
7  540-ws-training 2027-05-19 12:11:29             78
8  540-ws-training 2027-05-19 12:16:29             90
9  540-ws-training 2027-05-19 12:21:29             99


In [5]:
merged["hour"] = merged["timestamp"].dt.hour

merged["day_of_week"] = merged["timestamp"].dt.dayofweek

merged["is_weekend"] = (
    merged["day_of_week"] >= 5
).astype(int)

In [6]:
def get_time_period(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"

merged["time_period"] = merged["hour"].apply(get_time_period)

In [7]:
print(
    merged[[
        "timestamp",
        "hour",
        "day_of_week",
        "is_weekend",
        "time_period"
    ]].head(20)
)

             timestamp  hour  day_of_week  is_weekend time_period
0  2027-05-19 11:36:29    11            2           0     morning
1  2027-05-19 11:41:29    11            2           0     morning
2  2027-05-19 11:46:29    11            2           0     morning
3  2027-05-19 11:51:29    11            2           0     morning
4  2027-05-19 11:56:29    11            2           0     morning
5  2027-05-19 12:01:29    12            2           0   afternoon
6  2027-05-19 12:06:29    12            2           0   afternoon
7  2027-05-19 12:11:29    12            2           0   afternoon
8  2027-05-19 12:16:29    12            2           0   afternoon
9  2027-05-19 12:21:29    12            2           0   afternoon
10 2027-05-19 12:26:29    12            2           0   afternoon
11 2027-05-19 12:31:29    12            2           0   afternoon
12 2027-05-19 12:36:29    12            2           0   afternoon
13 2027-05-19 12:41:29    12            2           0   afternoon
14 2027-05

In [8]:
print(merged["time_period"].value_counts())

time_period
night        22310
morning      18479
afternoon    13602
evening      11144
Name: count, dtype: int64


In [9]:
merged = merged.sort_values(
    ["Patient", "timestamp"]
).reset_index(drop=True)

In [10]:
merged["glucose_prev_5min"] = (
    merged.groupby("Patient")["glucose_value"]
    .shift(1)
)

merged["glucose_prev_10min"] = (
    merged.groupby("Patient")["glucose_value"]
    .shift(2)
)

merged["glucose_prev_15min"] = (
    merged.groupby("Patient")["glucose_value"]
    .shift(3)
)

merged["glucose_prev_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .shift(6)
)

In [12]:
merged["glucose_change_5min"] = (
    merged["glucose_value"] - merged["glucose_prev_5min"]
)

merged["glucose_change_10min"] = (
    merged["glucose_value"] - merged["glucose_prev_10min"]
)

merged["glucose_change_15min"] = (
    merged["glucose_value"] - merged["glucose_prev_15min"]
)

merged["glucose_change_30min"] = (
    merged["glucose_value"] - merged["glucose_prev_30min"]
)

merged["glucose_rate_15min"] = (
    merged["glucose_change_15min"] / 15
)

merged["glucose_rate_30min"] = (
    merged["glucose_change_30min"] / 30
)

In [14]:
merged["glucose_rate_5min"] = (
    merged["glucose_change_5min"] / 5
)

merged["glucose_rate_15min"] = (
    merged["glucose_change_15min"] / 15
)

merged["glucose_rate_30min"] = (
    merged["glucose_change_30min"] / 30
)

In [15]:
print(
    merged[[
        "Patient",
        "timestamp",
        "glucose_value",
        "glucose_change_5min",
        "glucose_change_15min",
        "glucose_change_30min",
        "glucose_rate_5min",
        "glucose_rate_15min",
        "glucose_rate_30min"
    ]].head(15)
)

            Patient           timestamp  glucose_value  glucose_change_5min  \
0   540-ws-training 2027-05-19 11:36:29             76                  NaN   
1   540-ws-training 2027-05-19 11:41:29             72                 -4.0   
2   540-ws-training 2027-05-19 11:46:29             68                 -4.0   
3   540-ws-training 2027-05-19 11:51:29             65                 -3.0   
4   540-ws-training 2027-05-19 11:56:29             63                 -2.0   
5   540-ws-training 2027-05-19 12:01:29             66                  3.0   
6   540-ws-training 2027-05-19 12:06:29             71                  5.0   
7   540-ws-training 2027-05-19 12:11:29             78                  7.0   
8   540-ws-training 2027-05-19 12:16:29             90                 12.0   
9   540-ws-training 2027-05-19 12:21:29             99                  9.0   
10  540-ws-training 2027-05-19 12:26:29            110                 11.0   
11  540-ws-training 2027-05-19 12:31:29            1

In [16]:
print(
    merged[[
        "glucose_change_5min",
        "glucose_change_15min",
        "glucose_change_30min"
    ]].describe()
)

       glucose_change_5min  glucose_change_15min  glucose_change_30min
count         65529.000000          65517.000000          65499.000000
mean              0.011461              0.035136              0.069925
std               7.201349             16.482178             27.225224
min            -286.000000           -301.000000           -317.000000
25%              -3.000000             -8.000000            -14.000000
50%               0.000000             -1.000000             -1.000000
75%               3.000000              7.000000             13.000000
max             297.000000            310.000000            310.000000


Step 3 — Rolling Glucose Statistics

In [17]:
merged["glucose_rolling_mean_15min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

In [18]:
merged["glucose_rolling_mean_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(6, min_periods=1).mean())
)

In [19]:
merged["glucose_rolling_std_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(6, min_periods=2).std())
)

In [20]:
merged["glucose_min_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(6, min_periods=1).min())
)

merged["glucose_max_30min"] = (
    merged.groupby("Patient")["glucose_value"]
    .transform(lambda x: x.rolling(6, min_periods=1).max())
)

In [21]:
print(
    merged[[
        "Patient",
        "timestamp",
        "glucose_value",
        "glucose_rolling_mean_15min",
        "glucose_rolling_mean_30min",
        "glucose_rolling_std_30min",
        "glucose_min_30min",
        "glucose_max_30min"
    ]].head(15)
)

            Patient           timestamp  glucose_value  \
0   540-ws-training 2027-05-19 11:36:29             76   
1   540-ws-training 2027-05-19 11:41:29             72   
2   540-ws-training 2027-05-19 11:46:29             68   
3   540-ws-training 2027-05-19 11:51:29             65   
4   540-ws-training 2027-05-19 11:56:29             63   
5   540-ws-training 2027-05-19 12:01:29             66   
6   540-ws-training 2027-05-19 12:06:29             71   
7   540-ws-training 2027-05-19 12:11:29             78   
8   540-ws-training 2027-05-19 12:16:29             90   
9   540-ws-training 2027-05-19 12:21:29             99   
10  540-ws-training 2027-05-19 12:26:29            110   
11  540-ws-training 2027-05-19 12:31:29            121   
12  540-ws-training 2027-05-19 12:36:29            131   
13  540-ws-training 2027-05-19 12:41:29            137   
14  540-ws-training 2027-05-19 12:46:29            140   

    glucose_rolling_mean_15min  glucose_rolling_mean_30min  \
0        

In [22]:
print(
    merged[[
        "glucose_rolling_mean_15min",
        "glucose_rolling_mean_30min",
        "glucose_rolling_std_30min",
        "glucose_min_30min",
        "glucose_max_30min"
    ]].describe()
)

       glucose_rolling_mean_15min  glucose_rolling_mean_30min  \
count                65535.000000                65535.000000   
mean                   157.688739                  157.671140   
std                     60.650731                   60.175888   
min                     40.000000                   40.000000   
25%                    112.333333                  112.666667   
50%                    149.333333                  149.500000   
75%                    193.000000                  192.666667   
max                    400.000000                  400.000000   

       glucose_rolling_std_30min  glucose_min_30min  glucose_max_30min  
count               65529.000000       65535.000000       65535.000000  
mean                    7.080267         148.620294         166.764141  
std                     7.220368          58.662400          62.190581  
min                     0.000000          40.000000          40.000000  
25%                     2.732520         105.0000

Insulin Features

In [23]:
merged["bolus_given"] = (
    merged["bolus_dose"].notna()
).astype(int)

In [24]:
print(merged["bolus_given"].value_counts())

bolus_given
1    65294
0      241
Name: count, dtype: int64


In [25]:
merged["bolus_dose"] = pd.to_numeric(
    merged["bolus_dose"],
    errors="coerce"
)

merged["bolus_dose_filled"] = (
    merged["bolus_dose"].fillna(0)
)

In [26]:
merged["bolus_30min"] = (
    merged.groupby("Patient")["bolus_dose_filled"]
    .transform(
        lambda x: x.rolling(6, min_periods=1).sum()
    )
)

In [27]:
merged["bolus_60min"] = (
    merged.groupby("Patient")["bolus_dose_filled"]
    .transform(
        lambda x: x.rolling(12, min_periods=1).sum()
    )
)

In [28]:
print(
    merged[[
        "Patient",
        "timestamp",
        "bolus_dose",
        "bolus_given",
        "bolus_30min",
        "bolus_60min"
    ]].head(20)
)

            Patient           timestamp  bolus_dose  bolus_given  bolus_30min  \
0   540-ws-training 2027-05-19 11:36:29         0.8            1          0.8   
1   540-ws-training 2027-05-19 11:41:29         0.8            1          1.6   
2   540-ws-training 2027-05-19 11:46:29         0.8            1          2.4   
3   540-ws-training 2027-05-19 11:51:29         0.8            1          3.2   
4   540-ws-training 2027-05-19 11:56:29         0.8            1          4.0   
5   540-ws-training 2027-05-19 12:01:29         0.8            1          4.8   
6   540-ws-training 2027-05-19 12:06:29         0.8            1          4.8   
7   540-ws-training 2027-05-19 12:11:29         5.5            1          9.5   
8   540-ws-training 2027-05-19 12:16:29         5.5            1         14.2   
9   540-ws-training 2027-05-19 12:21:29         5.5            1         18.9   
10  540-ws-training 2027-05-19 12:26:29         5.5            1         23.6   
11  540-ws-training 2027-05-

In [29]:
print(
    merged[[
        "bolus_dose",
        "bolus_30min",
        "bolus_60min"
    ]].describe()
)

         bolus_dose   bolus_30min   bolus_60min
count  65294.000000  65535.000000  65535.000000
mean       6.792060     40.592538     81.162515
std        4.854745     28.774137     56.727741
min        0.100000      0.000000      0.000000
25%        3.000000     18.000000     36.000000
50%        5.900000     35.400000     70.800000
75%        9.300000     55.800000    111.600000
max       25.000000    150.000000    300.000000


In [30]:
print(merged["basal_value"].describe())

count    65535.000000
mean         1.067877
std          0.510238
min          0.400000
25%          0.450000
50%          1.100000
75%          1.500000
max          2.000000
Name: basal_value, dtype: float64


In [31]:
merged["basal_rolling_30min"] = (
    merged.groupby("Patient")["basal_value"]
    .transform(
        lambda x: x.rolling(6, min_periods=1).mean()
    )
)

In [32]:
merged["basal_rolling_60min"] = (
    merged.groupby("Patient")["basal_value"]
    .transform(
        lambda x: x.rolling(12, min_periods=1).mean()
    )
)

In [33]:
merged["temp_basal_active"] = (
    merged["temp_basal_value"].notna()
).astype(int)

In [34]:
merged["temp_basal_value"] = pd.to_numeric(
    merged["temp_basal_value"],
    errors="coerce"
)

In [35]:
merged["effective_basal"] = merged["basal_value"]

In [36]:
merged.loc[
    merged["temp_basal_value"].notna(),
    "effective_basal"
] = merged.loc[
    merged["temp_basal_value"].notna(),
    "temp_basal_value"
]

In [37]:
merged["basal_change"] = (
    merged["effective_basal"] -
    merged["basal_value"]
)

In [38]:
print(
    merged[[
        "timestamp",
        "basal_value",
        "temp_basal_value",
        "effective_basal",
        "basal_change",
        "temp_basal_active",
        "basal_rolling_30min",
        "basal_rolling_60min"
    ]].head(20)
)

             timestamp  basal_value  temp_basal_value  effective_basal  \
0  2027-05-19 11:36:29         0.95               NaN             0.95   
1  2027-05-19 11:41:29         0.95               NaN             0.95   
2  2027-05-19 11:46:29         0.95               NaN             0.95   
3  2027-05-19 11:51:29         0.95               NaN             0.95   
4  2027-05-19 11:56:29         0.95               NaN             0.95   
5  2027-05-19 12:01:29         0.95               NaN             0.95   
6  2027-05-19 12:06:29         0.95               NaN             0.95   
7  2027-05-19 12:11:29         0.95               NaN             0.95   
8  2027-05-19 12:16:29         0.95               NaN             0.95   
9  2027-05-19 12:21:29         0.95               NaN             0.95   
10 2027-05-19 12:26:29         0.95               NaN             0.95   
11 2027-05-19 12:31:29         0.95               NaN             0.95   
12 2027-05-19 12:36:29         0.95   

In [39]:
print(
    merged[[
        "basal_value",
        "temp_basal_value",
        "effective_basal",
        "basal_change",
        "basal_rolling_30min",
        "basal_rolling_60min"
    ]].describe()
)

        basal_value  temp_basal_value  effective_basal  basal_change  \
count  65535.000000      58609.000000     65535.000000  65535.000000   
mean       1.067877          0.139367         0.200766     -0.867111   
std        0.510238          0.403941         0.444413      0.775335   
min        0.400000          0.000000         0.000000     -2.000000   
25%        0.450000          0.000000         0.000000     -1.500000   
50%        1.100000          0.000000         0.000000     -1.000000   
75%        1.500000          0.000000         0.200000     -0.250000   
max        2.000000          2.340000         2.340000      1.700000   

       basal_rolling_30min  basal_rolling_60min  
count         65535.000000         65535.000000  
mean              1.067908             1.067944  
std               0.509748             0.509250  
min               0.400000             0.400000  
25%               0.450000             0.450000  
50%               1.100000             1.100000  
7

In [40]:
merged["temp_basal_active"] = (
    merged["temp_basal_value"] > 0
).astype(int)

In [41]:
merged["effective_basal"] = merged["basal_value"]

mask = merged["temp_basal_value"] > 0

merged.loc[mask, "effective_basal"] = (
    merged.loc[mask, "temp_basal_value"]
)

In [42]:
merged["basal_change"] = (
    merged["effective_basal"] -
    merged["basal_value"]
)

In [43]:
print(
    merged[[
        "basal_value",
        "temp_basal_value",
        "effective_basal",
        "basal_change",
        "temp_basal_active"
    ]].describe()
)

        basal_value  temp_basal_value  effective_basal  basal_change  \
count  65535.000000      58609.000000     65535.000000  65535.000000   
mean       1.067877          0.139367         1.105268      0.037390   
std        0.510238          0.403941         0.540113      0.275848   
min        0.400000          0.000000         0.020000     -0.600000   
25%        0.450000          0.000000         0.450000      0.000000   
50%        1.100000          0.000000         1.100000      0.000000   
75%        1.500000          0.000000         1.600000      0.000000   
max        2.000000          2.340000         2.340000      1.700000   

       temp_basal_active  
count       65535.000000  
mean            0.179980  
std             0.384174  
min             0.000000  
25%             0.000000  
50%             0.000000  
75%             0.000000  
max             1.000000  


Meal & Carbohydrate Features

In [44]:
merged["meal_carbs"] = pd.to_numeric(
    merged["meal_carbs"],
    errors="coerce"
)

In [45]:
merged["meal_event"] = (
    merged["meal_carbs"].notna() &
    (merged["meal_carbs"] > 0)
).astype(int)

In [46]:
merged["carbs_30min"] = (
    merged.groupby("Patient")["meal_carbs"]
    .transform(
        lambda x: x.fillna(0)
        .rolling(6, min_periods=1)
        .sum()
    )
)

In [47]:
merged["carbs_60min"] = (
    merged.groupby("Patient")["meal_carbs"]
    .transform(
        lambda x: x.fillna(0)
        .rolling(12, min_periods=1)
        .sum()
    )
)

In [48]:
print(
    merged[[
        "timestamp",
        "meal_type",
        "meal_carbs",
        "meal_event",
        "carbs_30min",
        "carbs_60min"
    ]].head(30)
)

             timestamp meal_type  meal_carbs  meal_event  carbs_30min  \
0  2027-05-19 11:36:29       NaN         NaN           0          0.0   
1  2027-05-19 11:41:29       NaN         NaN           0          0.0   
2  2027-05-19 11:46:29       NaN         NaN           0          0.0   
3  2027-05-19 11:51:29       NaN         NaN           0          0.0   
4  2027-05-19 11:56:29       NaN         NaN           0          0.0   
5  2027-05-19 12:01:29       NaN         NaN           0          0.0   
6  2027-05-19 12:06:29       NaN         NaN           0          0.0   
7  2027-05-19 12:11:29       NaN         NaN           0          0.0   
8  2027-05-19 12:16:29       NaN         NaN           0          0.0   
9  2027-05-19 12:21:29       NaN         NaN           0          0.0   
10 2027-05-19 12:26:29       NaN         NaN           0          0.0   
11 2027-05-19 12:31:29       NaN         NaN           0          0.0   
12 2027-05-19 12:36:29       NaN         NaN       

In [49]:
print(
    merged[[
        "meal_carbs",
        "carbs_30min",
        "carbs_60min"
    ]].describe()
)

        meal_carbs   carbs_30min   carbs_60min
count  64026.00000  65535.000000  65535.000000
mean      55.84650    327.298054    654.439002
std       30.47923    186.376428    370.187733
min        1.00000      0.000000      0.000000
25%       30.00000    180.000000    360.000000
50%       60.00000    360.000000    720.000000
75%       75.00000    434.500000    864.000000
max      162.00000    972.000000   1944.000000
